# Model Pipeline — Challenge ML 2026### Classification d'affiches de films en 5 genres`0 Animation` · `1 Blockbuster` · `2 Horreur` · `3 Comedie` · `4 Art & Essai`Trois approches independantes, puis fusion par vote souple :| # | Approche | Ce qu'elle regarde | Pre-entraine ? ||---|---|---|---|| 1 | CNN from scratch (PyTorch) | l'image brute | non — poids initialises aleatoirement || 2 | Gradient boosting sur features colorimetriques | couleur, texture, gradients (HOG) | non — descripteurs algorithmiques || 3 | Gradient boosting sur features de mise en page | geometrie du texte (MSER) | non — MSER est algorithmique |L'OCR a ete ecarte volontairement : Tesseract et EasyOCR sont des modeles pre-entraines, ce quele challenge interdit. L'approche 3 exploite donc la **geometrie** du texte sans le lire.

## 0. Environnement

In [ ]:
# !pip install torch torchvision opencv-python scikit-learn scikit-image pandas matplotlibimport os, random, time, json, warningsimport numpy as npimport pandas as pdimport cv2import matplotlib.pyplot as pltimport torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSamplerfrom torchvision import transformsfrom sklearn.model_selection import train_test_splitfrom sklearn.ensemble import HistGradientBoostingClassifierfrom sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrixfrom skimage.feature import hog, local_binary_patternfrom concurrent.futures import ThreadPoolExecutorwarnings.filterwarnings("ignore")

In [ ]:
# ----------------------- CONFIG -----------------------SEED         = 42IMAGE_FOLDER = os.path.join("train", "train")LABELS_CSV   = "train_labels.csv"# Aucun jeu de test n'est fourni : tout vient de train/train, decoupe en trois.KAGGLE_TEST_FOLDER = None       # a renseigner quand le jeu de test Kaggle sera disponibleVAL_SIZE, HOLDOUT_SIZE = 0.15, 0.15NUM_CLASSES  = 5CLASSES      = {0: "Animation", 1: "Blockbuster", 2: "Horreur", 3: "Comedie", 4: "Art&Essai"}# format portrait conserve (ratio ~0.70 constate dans l'EDA), pas de carre ecraseIMG_H, IMG_W = 192, 128BATCH_SIZE   = 64EPOCHS       = 40PATIENCE     = 8# /!\ Windows : un DataLoader avec num_workers > 0 et un Dataset defini dans un notebook# plante (multiprocessing en mode spawn). Sur Windows, laisser 0.NUM_WORKERS  = 0 if os.name == "nt" else 4random.seed(SEED); np.random.seed(SEED)torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Device : {DEVICE} | workers : {NUM_WORKERS}")if DEVICE.type == "cuda":    print(torch.cuda.get_device_name(0))

## 1. Donnees et split stratifie en troisAucun jeu de test n'est fourni : tout provient de `train/train`. Un simple decoupage train/valposerait un probleme, parce que la meme validation servirait a trois usages a la fois — arreterle CNN, valider les deux modeles de boosting, et calibrer les poids de fusion. Le score obtenuserait alors optimiste de plusieurs points, et l'ecart n'apparaitrait qu'au moment de lasoumission Kaggle.D'ou trois sous-ensembles :| Sous-ensemble | Part | Usage ||---|---|---|| **train** | 70 % | apprentissage des trois modeles || **val** | 15 % | early stopping du CNN, recherche des poids de fusion || **holdout** | 15 % | **touche une seule fois, a la fin** — estimation honnete du score Kaggle |Le split est fige par la seed et exporte dans `split.csv` : tous les membres du groupe doivententrainer sur exactement le meme decoupage, sinon vos scores ne sont pas comparables.

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)          # ajouter header=None si le csv n'a pas d'en-teteif labels_df.shape[1] != 2:    raise ValueError(f"{LABELS_CSV} : {labels_df.shape[1]} colonnes, 2 attendues")labels_df.columns = ["filename", "label"]labels_df["label"] = labels_df["label"].astype(int)fichiers_presents = set(os.listdir(IMAGE_FOLDER))manquants = set(labels_df.filename) - fichiers_presentsprint(f"{len(labels_df)} labels | {len(fichiers_presents)} fichiers | {len(manquants)} labels sans image")labels_df = labels_df[labels_df.filename.isin(fichiers_presents)].reset_index(drop=True)# premier decoupage : train vs (val + holdout)train_df, reste_df = train_test_split(    labels_df, test_size=VAL_SIZE + HOLDOUT_SIZE,    stratify=labels_df.label, random_state=SEED)# second decoupage : val vs holdoutval_df, holdout_df = train_test_split(    reste_df, test_size=HOLDOUT_SIZE / (VAL_SIZE + HOLDOUT_SIZE),    stratify=reste_df.label, random_state=SEED)train_df   = train_df.reset_index(drop=True)val_df     = val_df.reset_index(drop=True)holdout_df = holdout_df.reset_index(drop=True)pd.concat([train_df.assign(split="train"),           val_df.assign(split="val"),           holdout_df.assign(split="holdout")]).to_csv("split.csv", index=False)rep = pd.DataFrame({"train":   train_df.label.value_counts().sort_index(),                    "val":     val_df.label.value_counts().sort_index(),                    "holdout": holdout_df.label.value_counts().sort_index()})rep.index = [CLASSES[i] for i in rep.index]display(rep)# la classe minoritaire doit rester exploitable dans chaque sous-ensemblemini = rep.min().min()if mini < 20:    print(f"/!\ La plus petite cellule du tableau ne contient que {mini} images. "          f"Les metriques sur cette classe seront tres instables — envisager une "          f"validation croisee stratifiee plutot qu'un split simple.")counts = train_df.label.value_counts().sort_index().valuesCLASS_WEIGHTS = (len(train_df) / (NUM_CLASSES * counts)).astype(np.float32)print("class_weights :", dict(zip(CLASSES.values(), CLASS_WEIGHTS.round(3))))print(f"Baseline triviale (tout en classe majoritaire) : {counts.max()/counts.sum():.3f}")

## 2. Chargement des images`letterbox` redimensionne en conservant le ratio et complete par du noir. Un `resize` directvers un carre ecrase la composition verticale de 30 %, ce qui detruit la geometrie du titre etdu bloc de credits — precisement ce que l'approche 3 cherche a mesurer.

In [ ]:
def letterbox(img, h=IMG_H, w=IMG_W):    ih, iw = img.shape[:2]    s = min(h / ih, w / iw)    nh, nw = max(1, int(round(ih * s))), max(1, int(round(iw * s)))    canvas = np.zeros((h, w, 3), dtype=np.uint8)    top, left = (h - nh) // 2, (w - nw) // 2    canvas[top:top+nh, left:left+nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)    return canvasclass MoviePosterDataset(Dataset):    def __init__(self, df, image_folder, transform=None):        self.df = df.reset_index(drop=True)        self.image_folder = image_folder        self.transform = transform        self.echecs = []    def __len__(self):        return len(self.df)    def __getitem__(self, idx):        row = self.df.iloc[idx]        img = cv2.imread(os.path.join(self.image_folder, row["filename"]))        if img is None:            self.echecs.append(row["filename"])            img = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)        else:            img = letterbox(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))        if self.transform:            img = self.transform(img)        lab = int(row["label"]) if "label" in row.index and not pd.isna(row["label"]) else -1        return img, torch.tensor(lab, dtype=torch.long)

## 3. Normalisation calculee sur le corpusLes statistiques ImageNet ne sont pas utilisees : les affiches sont plus sombres et plus satureesque des photographies generalistes, et emprunter des constantes issues d'ImageNet dans unchallenge qui interdit le pre-entraine cree une ambiguite inutile dans le rendu.

In [ ]:
def compute_mean_std(df, image_folder, n=1500):    s, ss, k = np.zeros(3), np.zeros(3), 0    for f in df.filename.sample(min(n, len(df)), random_state=SEED):        img = cv2.imread(os.path.join(image_folder, f))        if img is None: continue        a = (cv2.cvtColor(letterbox(img), cv2.COLOR_BGR2RGB).astype(np.float32) / 255).reshape(-1, 3)        s += a.sum(0); ss += (a**2).sum(0); k += len(a)    mean = s / k    return mean, np.sqrt(np.maximum(ss / k - mean**2, 1e-8))t0 = time.time()MEAN, STD = compute_mean_std(train_df, IMAGE_FOLDER)print(f"mean = {MEAN.round(4).tolist()}")print(f"std  = {STD.round(4).tolist()}   ({time.time()-t0:.1f}s)")

## 4. AugmentationDeux transformations sont volontairement absentes :- **`RandomHorizontalFlip`** — une affiche contient du texte. Le miroir genere des formes  typographiques qui n'existent dans aucune affiche reelle, alors que la densite de texte est un  discriminant d'Art & Essai.- **`ColorJitter` fort** — l'EDA identifie luminosite, saturation et contraste comme features  discriminantes (dominante sombre = horreur, forte saturation = animation). Les randomiser a  ±20 % efface exactement le signal recherche. Reduit a 0.08.`RandomResizedCrop` remplace `RandomRotation`, qui creait des coins noirs que le reseau apprendcomme motif.

In [ ]:
norm = transforms.Normalize(mean=MEAN.tolist(), std=STD.tolist())transform_baseline = transforms.Compose([    transforms.ToPILImage(),    transforms.ToTensor(),    norm,])transform_augmented = transforms.Compose([    transforms.ToPILImage(),    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05),    transforms.RandomResizedCrop((IMG_H, IMG_W), scale=(0.80, 1.0), ratio=(0.62, 0.72)),    transforms.RandomAffine(degrees=4, translate=(0.03, 0.03), fill=0),    transforms.ToTensor(),    norm,    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),])# jamais d'augmentation ailleurs que sur le trainds_train   = MoviePosterDataset(train_df,   IMAGE_FOLDER, transform_augmented)ds_val     = MoviePosterDataset(val_df,     IMAGE_FOLDER, transform_baseline)ds_holdout = MoviePosterDataset(holdout_df, IMAGE_FOLDER, transform_baseline)common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,              pin_memory=(DEVICE.type == "cuda"))dl_train   = DataLoader(ds_train,   shuffle=True,  drop_last=True, **common)dl_val     = DataLoader(ds_val,     shuffle=False, **common)dl_holdout = DataLoader(ds_holdout, shuffle=False, **common)print(f"{len(ds_train)} train / {len(ds_val)} val / {len(ds_holdout)} holdout "      f"| {len(dl_train)} batches par epoque")

In [ ]:
# controle visuel de l'augmentation : verifier qu'aucune transformation ne denature l'affichexb, yb = next(iter(dl_train))inv = xb[:8].numpy().transpose(0, 2, 3, 1) * STD + MEANfig, axes = plt.subplots(1, 8, figsize=(16, 4))for ax, im, y in zip(axes, inv, yb[:8]):    ax.imshow(np.clip(im, 0, 1)); ax.set_title(CLASSES[int(y)], fontsize=8); ax.axis("off")plt.tight_layout(); plt.show()

## 5. Approche 1 — CNN from scratchLe point critique de l'architecture est la transition convolution → dense. Un`Flatten(128×28×28) → Linear(100352, 256)` represente 25,7 M parametres dans une seule couche,soit 99,6 % du reseau, pour 3 000 images d'entrainement : le modele memorise le train set.Ici, `AdaptiveAvgPool2d(1)` reduit le vecteur a 256 avant la couche dense. La capacite seconcentre dans les convolutions, la ou elle apprend des motifs plutot que des exemples.

In [ ]:
class CustomMovieCNN(nn.Module):    def __init__(self, num_classes=NUM_CLASSES, dropout=0.4):        super().__init__()        def bloc(cin, cout):            return nn.Sequential(                nn.Conv2d(cin, cout, 3, padding=1, bias=False),                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),                nn.Conv2d(cout, cout, 3, padding=1, bias=False),                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),                nn.MaxPool2d(2, 2),            )        self.features = nn.Sequential(            bloc(3, 32),      # 96 x 64            bloc(32, 64),     # 48 x 32            bloc(64, 128),    # 24 x 16            bloc(128, 256),   # 12 x 8            bloc(256, 256),   #  6 x 4        )        self.pool = nn.AdaptiveAvgPool2d(1)        self.classifier = nn.Sequential(            nn.Flatten(), nn.Dropout(dropout), nn.Linear(256, num_classes)        )    def forward(self, x):        return self.classifier(self.pool(self.features(x)))model = CustomMovieCNN()n_conv = sum(p.numel() for p in model.features.parameters())n_tot  = sum(p.numel() for p in model.parameters())print(f"Parametres totaux : {n_tot:,}")print(f"  dont convolutions : {n_conv:,} ({100*n_conv/n_tot:.1f} %)")print(f"  dont classifieur  : {n_tot-n_conv:,}")

In [ ]:
@torch.no_grad()def evaluate_model(model, dataloader):    model.eval()    preds, labs, probas = [], [], []    for images, labels in dataloader:        p = torch.softmax(model(images.to(DEVICE)), 1)        probas.append(p.cpu().numpy())        preds.extend(p.argmax(1).cpu().numpy())        labs.extend(labels.numpy())    probas = np.vstack(probas)    return (accuracy_score(labs, preds),            f1_score(labs, preds, average="macro"),            np.array(labs), probas)def train_model(model, train_loader, val_loader, class_weights=None,                epochs=EPOCHS, lr=1e-3, patience=PATIENCE, ckpt="best_cnn.pt"):    model.to(DEVICE)    w = None if class_weights is None else torch.tensor(class_weights, device=DEVICE)    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)    best_f1, best_ep, hist = -1.0, -1, []    for epoch in range(epochs):        model.train(); running = 0.0        for images, labels in train_loader:            images, labels = images.to(DEVICE), labels.to(DEVICE)            optimizer.zero_grad()            loss = criterion(model(images), labels)            loss.backward()            nn.utils.clip_grad_norm_(model.parameters(), 5.0)            optimizer.step()            running += loss.item()        scheduler.step()        acc, f1, _, _ = evaluate_model(model, val_loader)        hist.append({"epoch": epoch+1, "train_loss": running/len(train_loader),                     "val_acc": acc, "val_f1": f1})        flag = ""        if f1 > best_f1:            best_f1, best_ep = f1, epoch+1            torch.save(model.state_dict(), ckpt); flag = "  <-- meilleur"        print(f"Epoque [{epoch+1:2d}/{epochs}]  perte {running/len(train_loader):.4f}  "              f"val_acc {acc:.4f}  val_f1 {f1:.4f}{flag}")        if epoch + 1 - best_ep >= patience:            print(f"Early stopping — meilleur F1 {best_f1:.4f} a l'epoque {best_ep}")            break    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))    return model, pd.DataFrame(hist)

In [ ]:
t0 = time.time()model, hist = train_model(model, dl_train, dl_val, class_weights=CLASS_WEIGHTS)print(f"\nEntrainement : {(time.time()-t0)/60:.1f} min")fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))ax[0].plot(hist.epoch, hist.train_loss); ax[0].set_title("Perte (train)"); ax[0].set_xlabel("epoque")ax[1].plot(hist.epoch, hist.val_acc, label="accuracy")ax[1].plot(hist.epoch, hist.val_f1, label="F1 macro")ax[1].set_title("Validation"); ax[1].set_xlabel("epoque"); ax[1].legend()plt.tight_layout(); plt.show()

In [ ]:
acc_cnn, f1_cnn, y_val, PROBA_CNN = evaluate_model(model, dl_val)print(f"CNN — accuracy {acc_cnn:.4f} | F1 macro {f1_cnn:.4f}\n")print(classification_report(y_val, PROBA_CNN.argmax(1),                            target_names=[CLASSES[i] for i in range(NUM_CLASSES)], digits=3))cm = confusion_matrix(y_val, PROBA_CNN.argmax(1))plt.figure(figsize=(5.5, 4.5))plt.imshow(cm, cmap="Blues"); plt.colorbar()plt.xticks(range(NUM_CLASSES), CLASSES.values(), rotation=45, ha="right")plt.yticks(range(NUM_CLASSES), CLASSES.values())for i in range(NUM_CLASSES):    for j in range(NUM_CLASSES):        plt.text(j, i, cm[i, j], ha="center", color="white" if cm[i, j] > cm.max()/2 else "black")plt.title("Matrice de confusion — CNN"); plt.ylabel("vrai"); plt.xlabel("predit")plt.tight_layout(); plt.show()

## 6. Approche 2 — Gradient boosting sur descripteurs colorimetriquesHistogrammes HSV, statistiques de luminosite et de contraste, texture LBP, gradients HOG.Tous ces descripteurs sont algorithmiques : aucun poids appris sur un corpus externe.Les densites de contours haut/bas ont ete deliberement **retirees** de cette approche : ellesmesurent la meme chose que l'approche 3. Trois modeles qui regardent la meme informationfusionnent mal — l'interet du vote souple tient a la decorrelation de leurs erreurs.

In [ ]:
def features_couleur(img_bgr):    if img_bgr is None:        return np.zeros(64+32+32+10+10+540, dtype=np.float32)    img  = letterbox(img_bgr)    rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)    v_hist = cv2.calcHist([hsv], [2], None, [64], [0, 256]).ravel()    h_hist = cv2.calcHist([hsv], [0], None, [32], [0, 180]).ravel()    s_hist = cv2.calcHist([hsv], [1], None, [32], [0, 256]).ravel()    v_hist, h_hist, s_hist = [x / max(x.sum(), 1) for x in (v_hist, h_hist, s_hist)]    v, s = hsv[..., 2].astype(float), hsv[..., 1].astype(float)    scal = np.array([        rgb[..., 0].mean(), rgb[..., 1].mean(), rgb[..., 2].mean(),        gray.std(), gray.mean(),        s.mean(), s.std(),        (v < 60).mean(), (v > 200).mean(),        np.abs(rgb.astype(np.float32) - gray[..., None]).mean(),   # ~0 => noir et blanc    ], dtype=np.float32)    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")    lbp_hist = np.histogram(lbp, bins=10, range=(0, 10))[0].astype(np.float32)    lbp_hist /= max(lbp_hist.sum(), 1)    h_feat = hog(cv2.resize(gray, (64, 96)), orientations=9,                 pixels_per_cell=(16, 16), cells_per_block=(2, 2), feature_vector=True)    return np.concatenate([v_hist, h_hist, s_hist, scal, lbp_hist, h_feat]).astype(np.float32)def extraire(df, image_folder, fonction):    chemins = [os.path.join(image_folder, f) for f in df.filename]    with ThreadPoolExecutor(max_workers=(os.cpu_count() or 4) * 2) as ex:        return np.vstack(list(ex.map(lambda p: fonction(cv2.imread(p)), chemins)))t0 = time.time()Xc_tr = extraire(train_df,   IMAGE_FOLDER, features_couleur)Xc_va = extraire(val_df,     IMAGE_FOLDER, features_couleur)Xc_ho = extraire(holdout_df, IMAGE_FOLDER, features_couleur)print(f"{Xc_tr.shape[1]} features couleur/texture  ({time.time()-t0:.0f}s)")clf_couleur = HistGradientBoostingClassifier(    max_iter=500, learning_rate=0.06, l2_regularization=1.0,    class_weight="balanced", random_state=SEED,    early_stopping=True, validation_fraction=0.15,).fit(Xc_tr, train_df.label.values)PROBA_COULEUR    = clf_couleur.predict_proba(Xc_va)PROBA_COULEUR_HO = clf_couleur.predict_proba(Xc_ho)f1_couleur = f1_score(y_val, PROBA_COULEUR.argmax(1), average="macro")print(f"\nApproche couleur — F1 macro {f1_couleur:.4f}\n")print(classification_report(y_val, PROBA_COULEUR.argmax(1),                            target_names=[CLASSES[i] for i in range(NUM_CLASSES)], digits=3))

## 7. Approche 3 — Geometrie de la mise en page typographiqueOn ne lit pas le texte, on mesure sa geometrie : combien de glyphes, ou ils sont, et quelle estla hierarchie de leurs tailles.L'hypothese testee, propre a ce corpus : un **blockbuster** a un titre enorme plus un bloc decredits minuscule et dense en bas (le *billing block*), donc un ratio hauteur max / hauteur minextreme. Une affiche **d'art et essai** est saturee de texte de taille moyenne — lauriers defestivals, casting, citations de presse. Une affiche **d'animation** en porte peu, en groscaracteres.MSER (*Maximally Stable Extremal Regions*) isole les regions de niveau de gris stables, ce quesont les glyphes sur un fond quelconque. C'est un algorithme, pas un modele appris.

In [ ]:
H_ANALYSE = 512     # en dessous, la detection de texte n'est pas fiabledef _boites_texte(gray):    h, w = gray.shape    mser = cv2.MSER_create()    mser.setMinArea(max(12, int(0.00002 * h * w)))    mser.setMaxArea(int(0.02 * h * w))    mser.setDelta(5)    _, boxes = mser.detectRegions(gray)    retenues = []    for (x, y, bw, bh) in boxes:        if bw == 0 or bh == 0:            continue        if not (0.08 < bw / bh < 2.2):          # ni un trait ni une bande            continue        if not (0.008 < bh / h < 0.22):         # ni un pixel isole ni la moitie de l'affiche            continue        retenues.append((x, y, bw, bh))    if not retenues:        return np.empty((0, 4), dtype=int)    # MSER renvoie des regions imbriquees pour un meme glyphe : on deduplique    b = np.array(retenues)    b = b[np.argsort(-(b[:, 2] * b[:, 3]))]    gardees, occupe = [], np.zeros(gray.shape, dtype=bool)    for (x, y, bw, bh) in b:        zone = occupe[y:y+bh, x:x+bw]        if zone.size and zone.mean() > 0.6:            continue        occupe[y:y+bh, x:x+bw] = True        gardees.append((x, y, bw, bh))    return np.array(gardees)def _grouper_en_lignes(boites, tol=0.6):    if len(boites) == 0:        return []    centres = boites[:, 1] + boites[:, 3] / 2    hmed = np.median(boites[:, 3])    ordre = np.argsort(centres)    lignes, cur = [], [ordre[0]]    for i in ordre[1:]:        if abs(centres[i] - centres[cur[-1]]) <= tol * hmed:            cur.append(i)        else:            lignes.append(cur); cur = [i]    lignes.append(cur)    return lignesNOMS_LAYOUT = [    "nb_glyphes", "surface_texte", "hauteur_med", "hauteur_std", "hauteur_ratio_max_min",    "nb_lignes", "glyphes_par_ligne_med", "regularite_alignement",    "texte_bande_0", "texte_bande_1", "texte_bande_2", "texte_bande_3", "texte_bande_4",    "densite_bloc_credits", "hauteur_med_bas", "hauteur_max_relative",    "y_barycentre_texte", "x_dispersion_texte", "contraste_local_texte",    "symetrie_horizontale", "contours_haut", "contours_milieu", "contours_bas",    "nb_couleurs_dominantes",]def features_layout(img_bgr):    if img_bgr is None:        return np.zeros(len(NOMS_LAYOUT) + 16, dtype=np.float32)    h0, w0 = img_bgr.shape[:2]    sc = H_ANALYSE / h0    img = cv2.resize(img_bgr, (max(1, int(w0*sc)), H_ANALYSE), interpolation=cv2.INTER_AREA)    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)    h, w = gray.shape    b = _boites_texte(gray)    f = {k: 0.0 for k in NOMS_LAYOUT}    if len(b):        hauteurs = b[:, 3].astype(float)        aires = (b[:, 2] * b[:, 3]).astype(float)        cy = b[:, 1] + b[:, 3] / 2        cx = b[:, 0] + b[:, 2] / 2        f["nb_glyphes"]    = len(b)        f["surface_texte"] = aires.sum() / (h * w)        f["hauteur_med"]   = np.median(hauteurs) / h        f["hauteur_std"]   = hauteurs.std() / h        f["hauteur_ratio_max_min"] = hauteurs.max() / max(hauteurs.min(), 1.0)        f["hauteur_max_relative"]  = hauteurs.max() / h        lignes = _grouper_en_lignes(b)        f["nb_lignes"] = len(lignes)        f["glyphes_par_ligne_med"] = np.median([len(l) for l in lignes]) if lignes else 0.0        var = [hauteurs[l].std() / max(np.median(hauteurs[l]), 1.0) for l in lignes if len(l) > 2]        f["regularite_alignement"] = 1.0 / (1.0 + np.mean(var)) if var else 0.0        bandes = np.clip((cy / h * 5).astype(int), 0, 4)        for i in range(5):            f[f"texte_bande_{i}"] = aires[bandes == i].sum() / (h * w)        bas    = cy > 0.83 * h        # seuil absolu (2 % de la hauteur d'affiche) plutot que relatif a la mediane :        # si les credits dominent en nombre, la mediane devient elle-meme petite et le        # critere relatif ne detecte plus rien        petits = hauteurs < 0.02 * h        f["densite_bloc_credits"] = float((bas & petits).sum()) / len(b)        f["hauteur_med_bas"] = (np.median(hauteurs[bas]) / h) if bas.any() else 0.0        f["y_barycentre_texte"] = float(np.average(cy, weights=aires) / h)        f["x_dispersion_texte"] = float(cx.std() / w)        masque = np.zeros_like(gray, dtype=bool)        for (x, y, bw_, bh_) in b:            masque[y:y+bh_, x:x+bw_] = True        f["contraste_local_texte"] = float(gray[masque].std()) if masque.any() else 0.0    f["symetrie_horizontale"] = float(        1 - np.abs(gray.astype(np.float32) - np.fliplr(gray).astype(np.float32)).mean() / 255)    edges = cv2.Canny(gray, 80, 180)    t = h // 3    f["contours_haut"]   = edges[:t].mean() / 255    f["contours_milieu"] = edges[t:2*t].mean() / 255    f["contours_bas"]    = edges[2*t:].mean() / 255    petit = cv2.resize(img, (48, 72), interpolation=cv2.INTER_AREA).reshape(-1, 3).astype(np.float32)    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)    _, lab, _ = cv2.kmeans(petit, 8, None, crit, 3, cv2.KMEANS_PP_CENTERS)    parts = np.bincount(lab.ravel(), minlength=8) / len(lab)    f["nb_couleurs_dominantes"] = float((parts > 0.05).sum())    profil = cv2.resize(edges.mean(axis=1).astype(np.float32).reshape(-1, 1), (1, 16)).ravel() / 255    return np.concatenate([np.array([f[k] for k in NOMS_LAYOUT], np.float32), profil])

### Controle visuel avant d'entrainerMSER attrape parfois des yeux, des boutons ou des motifs repetitifs a la place des lettres. Sic'est le cas sur ces exemples, resserrer les filtres geometriques (`0.08 < ratio < 2.2`,`0.008 < hauteur relative < 0.22`) avant d'aller plus loin : ce reglage conditionne toutel'approche.

In [ ]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(4*NUM_CLASSES, 6))for c, ax in zip(range(NUM_CLASSES), axes):    f = train_df[train_df.label == c].filename.iloc[0]    img = cv2.imread(os.path.join(IMAGE_FOLDER, f))    sc = H_ANALYSE / img.shape[0]    img = cv2.resize(img, (int(img.shape[1]*sc), H_ANALYSE))    boites = _boites_texte(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))    vis = img.copy()    for (x, y, bw, bh) in boites:        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 1)    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))    ax.set_title(f"{CLASSES[c]}\n{len(boites)} glyphes", fontsize=9); ax.axis("off")plt.tight_layout(); plt.show()

In [ ]:
t0 = time.time()Xl_tr = extraire(train_df,   IMAGE_FOLDER, features_layout)Xl_va = extraire(val_df,     IMAGE_FOLDER, features_layout)Xl_ho = extraire(holdout_df, IMAGE_FOLDER, features_layout)print(f"{Xl_tr.shape[1]} features de layout  ({time.time()-t0:.0f}s)")# les features separent-elles reellement les classes ? tableau a reprendre dans le rapportdiag = pd.DataFrame(Xl_tr[:, :len(NOMS_LAYOUT)], columns=NOMS_LAYOUT)diag["classe"] = [CLASSES[i] for i in train_df.label.values]cles = ["nb_glyphes", "surface_texte", "hauteur_ratio_max_min", "nb_lignes",        "densite_bloc_credits", "y_barycentre_texte", "hauteur_max_relative"]display(diag.groupby("classe")[cles].mean().round(3))fig, axes = plt.subplots(2, 4, figsize=(17, 7))for c, ax in zip(cles, axes.ravel()):    diag.boxplot(column=c, by="classe", ax=ax, grid=False)    ax.set_title(c, fontsize=9); ax.set_xlabel("")    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, fontsize=7)axes.ravel()[-1].axis("off")plt.suptitle(""); plt.tight_layout(); plt.show()

In [ ]:
clf_layout = HistGradientBoostingClassifier(    max_iter=500, learning_rate=0.06, l2_regularization=1.0,    class_weight="balanced", random_state=SEED,    early_stopping=True, validation_fraction=0.15,).fit(Xl_tr, train_df.label.values)PROBA_LAYOUT    = clf_layout.predict_proba(Xl_va)PROBA_LAYOUT_HO = clf_layout.predict_proba(Xl_ho)f1_layout = f1_score(y_val, PROBA_LAYOUT.argmax(1), average="macro")print(f"Approche layout — F1 macro {f1_layout:.4f}\n")print(classification_report(y_val, PROBA_LAYOUT.argmax(1),                            target_names=[CLASSES[i] for i in range(NUM_CLASSES)], digits=3))

## 8. Fusion par vote soupleLes poids sont cherches par grille **sur la validation**, puis le resultat est mesure **une seulefois sur le holdout**, qui n'a servi a aucune decision jusqu'ici. C'est cette derniere valeur quiapproxime honnetement ce que rendra Kaggle : l'ecart entre le F1 de validation et celui duholdout mesure exactement le sur-ajustement introduit par la selection.Avant de fusionner, la correlation des erreurs dit si l'operation peut rapporter quelque chose :si les trois modeles se trompent sur les memes images, le vote ne corrigera rien.

In [ ]:
erreurs = pd.DataFrame({    "CNN":     (PROBA_CNN.argmax(1)     != y_val).astype(int),    "couleur": (PROBA_COULEUR.argmax(1) != y_val).astype(int),    "layout":  (PROBA_LAYOUT.argmax(1)  != y_val).astype(int),})print("Taux d'erreur :", erreurs.mean().round(3).to_dict())print("\nCorrelation des erreurs (bas = modeles complementaires) :")display(erreurs.corr().round(3))print(f"\nImages ratees par les 3 : {(erreurs.sum(1) == 3).sum()} / {len(y_val)}")print(f"Images reussies par au moins un : {(erreurs.sum(1) < 3).sum()} / {len(y_val)}")

In [ ]:
from itertools import productprobas = [PROBA_CNN, PROBA_COULEUR, PROBA_LAYOUT]noms   = ["CNN", "couleur", "layout"]best_w, best_f1 = None, -1for w in product(np.arange(0, 1.05, 0.05), repeat=3):    if sum(w) == 0:        continue    w = np.array(w) / sum(w)    f1 = f1_score(y_val, sum(wi*pi for wi, pi in zip(w, probas)).argmax(1), average="macro")    if f1 > best_f1:        best_w, best_f1 = w, f1print("F1 macro individuels :", dict(zip(noms, [round(f1_cnn,4), round(f1_couleur,4), round(f1_layout,4)])))print(f"Poids optimaux       : {dict(zip(noms, best_w.round(3)))}")print(f"F1 macro fusionne    : {best_f1:.4f}  (+{best_f1-max(f1_cnn,f1_couleur,f1_layout):.4f})\n")PROBA_FUSION = sum(wi*pi for wi, pi in zip(best_w, probas))print(classification_report(y_val, PROBA_FUSION.argmax(1),                            target_names=[CLASSES[i] for i in range(NUM_CLASSES)], digits=3))

## 8bis. Evaluation finale sur le holdoutCette cellule ne doit etre executee **qu'une seule fois**, une fois tous les choix arretes(architecture, augmentation, seuils MSER, poids de fusion). Chaque re-execution suivie d'unreglage transforme le holdout en seconde validation et lui fait perdre sa valeur.

In [ ]:
acc_ho_cnn, f1_ho_cnn, y_ho, PROBA_CNN_HO = evaluate_model(model, dl_holdout)probas_ho = [PROBA_CNN_HO, PROBA_COULEUR_HO, PROBA_LAYOUT_HO]PROBA_FUSION_HO = sum(w*p for w, p in zip(best_w, probas_ho))resume = pd.DataFrame({    "F1_val": [f1_cnn, f1_couleur, f1_layout, best_f1],    "F1_holdout": [        f1_ho_cnn,        f1_score(y_ho, PROBA_COULEUR_HO.argmax(1), average="macro"),        f1_score(y_ho, PROBA_LAYOUT_HO.argmax(1),  average="macro"),        f1_score(y_ho, PROBA_FUSION_HO.argmax(1),  average="macro"),    ],    "accuracy_holdout": [        acc_ho_cnn,        accuracy_score(y_ho, PROBA_COULEUR_HO.argmax(1)),        accuracy_score(y_ho, PROBA_LAYOUT_HO.argmax(1)),        accuracy_score(y_ho, PROBA_FUSION_HO.argmax(1)),    ],}, index=["CNN", "couleur", "layout", "fusion"]).round(4)resume["ecart_val_holdout"] = (resume.F1_val - resume.F1_holdout).round(4)display(resume)print("\nL'ecart val -> holdout mesure le sur-ajustement des choix faits sur la validation.")print("S'il depasse 0.05 sur la fusion, les poids sont sur-ajustes : elargir le pas de la")print("grille de recherche, ou se rabattre sur une moyenne simple des trois modeles.\n")print(classification_report(y_ho, PROBA_FUSION_HO.argmax(1),                            target_names=[CLASSES[i] for i in range(NUM_CLASSES)], digits=3))cm = confusion_matrix(y_ho, PROBA_FUSION_HO.argmax(1))plt.figure(figsize=(5.5, 4.5))plt.imshow(cm, cmap="Blues"); plt.colorbar()plt.xticks(range(NUM_CLASSES), CLASSES.values(), rotation=45, ha="right")plt.yticks(range(NUM_CLASSES), CLASSES.values())for i in range(NUM_CLASSES):    for j in range(NUM_CLASSES):        plt.text(j, i, cm[i, j], ha="center", color="white" if cm[i, j] > cm.max()/2 else "black")plt.title("Fusion — holdout"); plt.ylabel("vrai"); plt.xlabel("predit")plt.tight_layout(); plt.show()with open("poids_fusion.json", "w") as fp:    json.dump({"poids": best_w.tolist(), "modeles": noms,               "f1_val": float(best_f1),               "f1_holdout": float(resume.loc["fusion", "F1_holdout"])}, fp, indent=2)

## 9. Soumission KaggleDeux etapes distinctes.**Reentrainement sur la totalite des donnees.** Une fois les hyperparametres arretes, les 30 %mis de cote pour la validation et le holdout representent environ 1 100 affiches inutilisees.Les reintegrer est un gain net, en particulier pour la classe minoritaire. Le nombre d'epoquesn'est plus determine par early stopping, puisqu'il n'y a plus de validation : il est fige a lameilleure epoque trouvee precedemment.**Prediction.** A lancer quand le jeu de test Kaggle sera disponible : renseigner`KAGGLE_TEST_FOLDER`. Verifier le format de soumission attendu par la competition, les noms decolonnes et l'ordre des lignes varient d'un challenge a l'autre.

In [ ]:
MEILLEURE_EPOQUE = int(hist.loc[hist.val_f1.idxmax(), "epoch"])print(f"Meilleure epoque observee : {MEILLEURE_EPOQUE}")def reentrainer_sur_tout(epochs=None):    epochs = epochs or MEILLEURE_EPOQUE    tout = pd.concat([train_df, val_df, holdout_df]).reset_index(drop=True)    print(f"Reentrainement sur {len(tout)} images, {epochs} epoques")    cnt = tout.label.value_counts().sort_index().values    poids = torch.tensor((len(tout) / (NUM_CLASSES * cnt)).astype(np.float32), device=DEVICE)    dl = DataLoader(MoviePosterDataset(tout, IMAGE_FOLDER, transform_augmented),                    batch_size=BATCH_SIZE, shuffle=True, drop_last=True,                    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))    m = CustomMovieCNN().to(DEVICE)    crit = nn.CrossEntropyLoss(weight=poids, label_smoothing=0.05)    opt  = optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)    for ep in range(epochs):        m.train(); run = 0.0        for x, y in dl:            x, y = x.to(DEVICE), y.to(DEVICE)            opt.zero_grad()            loss = crit(m(x), y)            loss.backward()            nn.utils.clip_grad_norm_(m.parameters(), 5.0); opt.step()            run += loss.item()        sch.step()        print(f"  epoque {ep+1}/{epochs}  perte {run/len(dl):.4f}")    torch.save(m.state_dict(), "final_cnn.pt")    Xc = np.vstack([Xc_tr, Xc_va, Xc_ho])    Xl = np.vstack([Xl_tr, Xl_va, Xl_ho])    y_tout = np.concatenate([train_df.label.values, val_df.label.values, holdout_df.label.values])    hp = dict(max_iter=500, learning_rate=0.06, l2_regularization=1.0,              class_weight="balanced", random_state=SEED,              early_stopping=True, validation_fraction=0.15)    return m, HistGradientBoostingClassifier(**hp).fit(Xc, y_tout), \              HistGradientBoostingClassifier(**hp).fit(Xl, y_tout)# decommenter le jour de la soumission# model_final, clf_couleur_final, clf_layout_final = reentrainer_sur_tout()

In [ ]:
def predire_kaggle(test_folder, noms_fichiers=None,                   m=None, c_col=None, c_lay=None, sortie="submission.csv"):    m     = m     if m     is not None else model    c_col = c_col if c_col is not None else clf_couleur    c_lay = c_lay if c_lay is not None else clf_layout    if noms_fichiers is None:        noms_fichiers = sorted(f for f in os.listdir(test_folder)                               if f.lower().endswith((".jpg", ".jpeg", ".png")))    df_test = pd.DataFrame({"filename": noms_fichiers, "label": -1})    print(f"{len(df_test)} images a predire")    dl = DataLoader(MoviePosterDataset(df_test, test_folder, transform_baseline),                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)    m.eval().to(DEVICE)    with torch.no_grad():        p_cnn = np.vstack([torch.softmax(m(x.to(DEVICE)), 1).cpu().numpy() for x, _ in dl])    p_col = c_col.predict_proba(extraire(df_test, test_folder, features_couleur))    p_lay = c_lay.predict_proba(extraire(df_test, test_folder, features_layout))    fusion = sum(w*p for w, p in zip(best_w, [p_cnn, p_col, p_lay]))    out = pd.DataFrame({"filename": noms_fichiers, "label": fusion.argmax(1)})    out.to_csv(sortie, index=False)      # adapter les noms de colonnes au format Kaggle    print(f"-> {sortie}")    # garde-fou : une distribution predite tres eloignee de celle du train signale un probleme    ref = labels_df.label.value_counts(normalize=True).sort_index()    obs = out.label.value_counts(normalize=True).sort_index().reindex(ref.index, fill_value=0)    display(pd.DataFrame({"train": ref.round(3), "predit": obs.round(3)}))    return out, {"cnn": p_cnn, "couleur": p_col, "layout": p_lay, "fusion": fusion}if KAGGLE_TEST_FOLDER and os.path.isdir(KAGGLE_TEST_FOLDER):    submission, probas_test = predire_kaggle(KAGGLE_TEST_FOLDER)    display(submission.head())else:    print("KAGGLE_TEST_FOLDER non renseigne — a executer quand le jeu de test sera disponible.")

## 10. Recapitulatif| Element | Valeur ||---|---|| Decoupage | 70 / 15 / 15 stratifie, seed 42, exporte dans `split.csv` || Metrique | F1 macro — le desequilibre rend l'accuracy trompeuse || Selection des choix | validation : early stopping du CNN, poids de fusion || Estimation finale | holdout, evalue une seule fois || Fichiers produits | `split.csv`, `best_cnn.pt`, `poids_fusion.json`, `submission.csv` |**Points a reprendre dans le rapport**- le decoupage en trois : sans jeu de test fourni, employer la meme validation pour arreter le  CNN et pour calibrer les poids de fusion produit un score optimiste. L'ecart mesure entre F1  de validation et F1 de holdout chiffre exactement ce biais.- l'accuracy seule est trompeuse : la classe Blockbuster represente environ 4,5 % du corpus, un  modele qui l'ignore totalement conserve une accuracy elevee. D'ou le F1 macro et les poids de  classe dans la fonction de cout.- justifier l'absence de `RandomHorizontalFlip` et de `ColorJitter` fort par les conclusions de  l'EDA : ces transformations detruisent les features identifiees comme discriminantes.- justifier l'abandon de l'OCR par la contrainte sur les modeles pre-entraines, et decrire  l'approche 3 comme une analyse structurelle de la mise en page, pas comme du NLP.- si l'ecart validation / holdout depasse 0.05 sur la fusion, preferer une moyenne simple des  trois modeles : moins performante en validation, plus robuste sur des donnees nouvelles.